In [ ]:
# ============================================================
# Notebook 03 - Phase 1: MLP Baseline
# CS-419 Deep Learning Project - Speech Emotion Recognition
# ============================================================
# Experiments in this notebook:
#   A. No regularization MLP (observe overfitting)
#   B. MLP with dropout + L2
#   C. Optimizer comparison: SGD vs Adam
#   D. Activation function comparison: ReLU vs Tanh vs Sigmoid
# ============================================================

import sys
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt

sys.path.append("../src")

from models import build_mlp_baseline
from train import train_model, plot_history, compare_histories
from evaluate import evaluate_model, plot_model_comparison_bar
from data_loader import get_class_weights, split_dataset, load_tess_paths
from utils import set_seeds, save_results_csv

set_seeds(42)
os.makedirs("results/plots", exist_ok=True)
os.makedirs("results/models", exist_ok=True)

# ---- Cell 1: Load cached flat features ----

print("Loading cached MFCC features...")
train_data = np.load("results/cache/train_flat.npz")
val_data   = np.load("results/cache/val_flat.npz")
test_data  = np.load("results/cache/test_flat.npz")

X_train, y_train = train_data["X"], train_data["y"]
X_val,   y_val   = val_data["X"],   val_data["y"]
X_test,  y_test  = test_data["X"],  test_data["y"]

# Load normalizer
with open("results/cache/flat_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}    y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")

# ---- Cell 2: Compute class weights ----

DATA_ROOT = "data/TESS Toronto emotional speech set data"
df = load_tess_paths(DATA_ROOT)
df_train, _, _ = split_dataset(df)
class_weights = get_class_weights(df_train)

# ---- Cell 3: Experiment A - Baseline MLP, no regularization ----
# This model will overfit. We show this deliberately.

print("\n=== Experiment A: MLP, no regularization ===")
model_a = build_mlp_baseline(
    input_dim=X_train.shape[1],
    hidden_units=(256, 128, 64),
    dropout_rate=0.0,
    l2_reg=0.0,
    optimizer="adam",
    learning_rate=1e-3,
)
model_a.summary()

hist_a = train_model(
    model_a, X_train, y_train, X_val, y_val,
    model_name="MLP_no_reg",
    epochs=80,
    batch_size=32,
    class_weights=None,
)
plot_history(hist_a, "MLP_no_reg")
res_a = evaluate_model(model_a, X_test, y_test, model_name="MLP_no_reg")

# ---- Cell 4: Experiment B - MLP with dropout + L2 + class weights ----

print("\n=== Experiment B: MLP with Dropout + L2 ===")
model_b = build_mlp_baseline(
    input_dim=X_train.shape[1],
    hidden_units=(256, 128, 64),
    dropout_rate=0.35,
    l2_reg=1e-4,
    optimizer="adam",
    learning_rate=1e-3,
)

hist_b = train_model(
    model_b, X_train, y_train, X_val, y_val,
    model_name="MLP_regularized",
    epochs=80,
    batch_size=32,
    class_weights=class_weights,
)
plot_history(hist_b, "MLP_regularized")
res_b = evaluate_model(model_b, X_test, y_test, model_name="MLP_regularized")

# ---- Cell 5: Experiment C - Optimizer comparison (Adam vs SGD) ----

print("\n=== Experiment C: Adam vs SGD ===")

model_sgd = build_mlp_baseline(
    input_dim=X_train.shape[1],
    hidden_units=(256, 128, 64),
    dropout_rate=0.35,
    l2_reg=1e-4,
    optimizer="sgd",
    learning_rate=0.01,
)
hist_sgd = train_model(
    model_sgd, X_train, y_train, X_val, y_val,
    model_name="MLP_SGD",
    epochs=80,
    batch_size=32,
    class_weights=class_weights,
)
res_sgd = evaluate_model(model_sgd, X_test, y_test, model_name="MLP_SGD")

compare_histories(
    {"MLP-Adam": hist_b, "MLP-SGD": hist_sgd},
    metric="val_accuracy"
)

# ---- Cell 6: Experiment D - Activation functions ----
# Swap out ReLU for Tanh (requires custom model change)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

def build_mlp_with_activation(activation, input_dim=96):
    inp = keras.Input(shape=(input_dim,))
    x = inp
    for units in [256, 128, 64]:
        x = layers.Dense(units, kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
        x = layers.Dropout(0.35)(x)
    out = layers.Dense(7, activation="softmax")(x)
    model = keras.Model(inp, out, name=f"MLP_{activation}")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

activation_results = {}
for act in ["relu", "tanh", "elu"]:
    print(f"\n--- Activation: {act} ---")
    m = build_mlp_with_activation(act, X_train.shape[1])
    h = train_model(m, X_train, y_train, X_val, y_val,
                    model_name=f"MLP_{act}", epochs=60, batch_size=32,
                    class_weights=class_weights)
    activation_results[f"MLP-{act.upper()}"] = h

compare_histories(activation_results, metric="val_accuracy")

# ---- Cell 7: Save best MLP and results ----

model_b.save("results/models/mlp_best.keras")
print("Best MLP model saved.")

summary = [
    {"model": "MLP (no reg)",    "accuracy": res_a["accuracy"], "macro_f1": res_a["macro_f1"]},
    {"model": "MLP (reg+Adam)",  "accuracy": res_b["accuracy"], "macro_f1": res_b["macro_f1"]},
    {"model": "MLP (SGD)",       "accuracy": res_sgd["accuracy"], "macro_f1": res_sgd["macro_f1"]},
]
df_results = save_results_csv(summary, "results/mlp_experiment_results.csv")
print(df_results.to_string(index=False))

print("\n=== Phase 1 Complete ===")
print(f"Best MLP Test Accuracy : {res_b['accuracy']*100:.2f}%")
print(f"Best MLP Macro F1      : {res_b['macro_f1']:.4f}")